# OpenShorts on free Google Colab (GPU)

Runs [OpenShorts](https://github.com/mutonby/openshorts) (long video → vertical viral clips with subtitles) on a free Colab T4 GPU and gives you a public dashboard link.

1. **Runtime → Change runtime type → T4 GPU**
2. Get a free Gemini key at https://aistudio.google.com/apikey and paste it in cell 2 (or enter it later in the dashboard under Settings)
3. Run all cells in order. The last cell prints a `https://….trycloudflare.com` link; open it.

Keep the tab open while you work: Colab stops everything when the session ends, so download your clips before closing.

In [ ]:
#@title 1. Install system packages, Node 20 and cloudflared
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU (CPU works too, just slower)"
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-liberation fonts-noto-color-emoji > /dev/null
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null && apt-get -qq install -y nodejs > /dev/null
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!node -v && ffmpeg -version | head -1 && cloudflared --version

In [ ]:
#@title 2. Clone OpenShorts and write .env
GEMINI_API_KEY = ""  #@param {type:"string"}
WHISPER_MODEL = "large-v3-turbo"  #@param ["small", "medium", "large-v3-turbo"]

import os, subprocess
if not os.path.isdir("/content/openshorts"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mutonby/openshorts.git", "/content/openshorts"], check=True)
os.chdir("/content/openshorts")

gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
env = {
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "WHISPER_MODEL": WHISPER_MODEL if gpu else "small",
    "WHISPER_DEVICE": "cuda" if gpu else "cpu",
    "WHISPER_COMPUTE": "float16" if gpu else "int8",
    "FFMPEG_ENCODER": "auto",
}
with open(".env", "w") as f:
    f.writelines(f"{k}={v}\n" for k, v in env.items())
print("GPU:", gpu); print(open(".env").read().replace(GEMINI_API_KEY or "\0", "***"))

In [ ]:
#@title 3. Install Python + dashboard dependencies (~5-8 min)
%cd /content/openshorts
!pip install -q -r requirements.txt
!pip install -q --upgrade --pre "yt-dlp[default]"
!mkdir -p /usr/local/share/fonts/openshorts && cp fonts/*.ttf /usr/local/share/fonts/openshorts/ && cp fonts/openshorts-fontmap.conf /etc/fonts/conf.d/60-openshorts.conf && fc-cache -f
# Let the Vite dev server answer on the Cloudflare tunnel hostname.
!grep -q trycloudflare dashboard/vite.config.js || sed -i "s/allowedHosts: \[/allowedHosts: ['.trycloudflare.com', /" dashboard/vite.config.js
!cd dashboard && npm install --no-audit --no-fund --loglevel=error

In [ ]:
#@title 4. Start backend, dashboard and public link
import subprocess, time, re, os
os.chdir("/content/openshorts")
os.makedirs("output", exist_ok=True)

backend = subprocess.Popen("uvicorn app:app --host 0.0.0.0 --port 8000 > /content/backend.log 2>&1", shell=True)
frontend = subprocess.Popen("cd dashboard && VITE_PROXY_TARGET=http://localhost:8000 npx vite --host 0.0.0.0 --port 5173 > /content/frontend.log 2>&1", shell=True)
tunnel = subprocess.Popen("cloudflared tunnel --url http://localhost:5173 --no-autoupdate > /content/tunnel.log 2>&1", shell=True)

url = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0); break
print("Dashboard:", url or "not ready, check /content/tunnel.log")
print("Backend health:", subprocess.run("curl -s localhost:8000/health", shell=True, capture_output=True, text=True).stdout or "starting... run cell 5 in a minute")

In [ ]:
#@title 5. Logs (re-run to refresh)
!tail -n 25 /content/backend.log
!echo ---- && tail -n 5 /content/frontend.log

In [ ]:
#@title 6. Download all generated clips as a zip
!cd /content/openshorts && zip -qr /content/openshorts_clips.zip output
from google.colab import files
files.download("/content/openshorts_clips.zip")